In [1]:
from dotenv import load_dotenv
load_dotenv()

True

https://console.groq.com/playground?model=openai/gpt-oss-20b

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [4]:
llm("ဟယ်လို နေကောင်းလား")

'ဟယ်လို နေကောင်းလား။ သင့်အား ခင်မင်ပါတယ်။'

In [5]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

I'm excited you're interested in joining. However, I need a bit more information from you. Could you please tell me what course you're referring to? Is it an online course, a university course, or something else? Additionally, what's the current date and enrollment deadline for the course? This will help me provide a more accurate answer to your question.


Adding context manually

In [6]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

In [7]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


## Retrieval plus generation

```
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

Three components:

- search
- prompt
- LLM

Architecture
![Alt Text](notes/rag_flow_1.png)
 

In [9]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
for course in courses_raw:
    print(course)

{'course': 'machine-learning-zoomcamp', 'course_name': 'ML Zoomcamp', 'path': '/json/machine-learning-zoomcamp.json', 'questions_count': 472}
{'course': 'llm-zoomcamp', 'course_name': 'LLM Zoomcamp', 'path': '/json/llm-zoomcamp.json', 'questions_count': 79}
{'course': 'data-engineering-zoomcamp', 'course_name': 'Data Engineering Zoomcamp', 'path': '/json/data-engineering-zoomcamp.json', 'questions_count': 402}
{'course': 'mlops-zoomcamp', 'course_name': 'MLOps Zoomcamp', 'path': '/json/mlops-zoomcamp.json', 'questions_count': 255}


In [11]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [12]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

NOTE: this data is already cleaned. In reality  we may need to scrape websites, parse PDFs, and clean and chunk documents. 

### Search
- Indexing with minsearch
    - Searching matters because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow, and the model would get confused with too much data. Instead, search finds the most relevant documents to send.
    - other heavier options for search : Apache Lucene, Elasticsearch, Solr

In [13]:
from minsearch import Index

index = Index(
    # Text fields are the fields you search through. When you type a query, the search engine looks at these fields, 
    # tokenizes them (splits into words, lowercases, removes stop words), and ranks the results by relevance. 
    text_fields=['question', 'section', 'answer'],
    # Keyword fields are for exact matching. Make the result come from the specific course
    keyword_fields=['course']
)

index.fit(documents)

In [14]:
question = 'I just discovered the course. Can I join now?'

search_results = index.search(
    question,

    # boost_dict to give the question field more weight (2.0 instead of the default 1.0) 
    # section less weight (0.5)
    #  if the query words appear in the question field, that's a stronger signal than if they appear in the section name.
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
[doc['question'] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

### Building the Prompt

the prompt consists of two parts:

- Instructions (also called system prompt): this tells the LLM how to behave. It never changes - it's the same for every request.
- User prompt: this changes with every request. It contains the actual question and the retrieved context.

In [16]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

grounds the answer in our data and reduces hallucinations.

In [17]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [18]:
print(USER_PROMPT_TEMPLATE.format(question = "how are you", context = "this is a context"))


Question:
how are you

Context:
this is a context



### Building the context

In [19]:
# context is a formatted string with all the search results
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

### Building the prompt

In [20]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [22]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

## The LLM

### Sending the prompt to the LLM

In [23]:
response = client.responses.create(
    model='openai/gpt-oss-20b', # reasoning model
    input=prompt
)


In [24]:
response.output[0]

ResponseReasoningItem(id='resp_01kt3w6by9er7vwhpgmtvzfexr', summary=[], type='reasoning', content=[Content(text='The user: "Question: I just discovered the course. Can I join now?" We have context. We need to answer based on policy: "I just discovered the course. Can I still join?" The answer: Yes, but to get certificate, submit project while submissions accepted. So respond with that.', type='reasoning_text')], encrypted_content=None, status='completed')

In [25]:
# Internal reasoning step
response.output[0].content[0].text

'The user: "Question: I just discovered the course. Can I join now?" We have context. We need to answer based on policy: "I just discovered the course. Can I still join?" The answer: Yes, but to get certificate, submit project while submissions accepted. So respond with that.'

In [26]:
# Final generated answer
response.output_text

'Yes—you can still join, but there’s a catch if you want a certificate.  \nYou’ll need to submit your Capstone project while the course is still accepting submissions. Once the submission window closes, the peer‑review process starts and the certificate is only awarded to those who finish during a live cohort. If you’re just looking to learn, feel free to start the modules and submit homework as soon as the form is open.'

In [27]:
response.usage

ResponseUsage(input_tokens=399, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=159, output_tokens_details=OutputTokensDetails(reasoning_tokens=63), total_tokens=558)

In [28]:
len(response.output_text.split())

72

### Calculating the price

e.g. gpt-5.4-mini:

- Input: 0.75 per million tokens
- Output: 4.50 per million tokens

In [29]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00101475

### Message history

- It starts with a hidden system prompt that tells the LLM how to behave. 
- After that, your messages and the LLM's replies alternate. 
- The LLM needs the full history to continue the conversation.

In [30]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},  # tell the system how to behave
    {'role': 'user', 'content': prompt}  # actual prompt with context and question
]

response = client.responses.create(
    model='openai/gpt-oss-20b',
    input=message_history
)

In [31]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

## Full RAG

In [32]:
def search(query, num_results=5):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'course': "machine-learning-zoomcamp"}

        return index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

In [33]:
def rag(query, model='openai/gpt-oss-20b'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [34]:
answer = rag('I just discovered the course. Can I join now?')
print(answer)

Yes – you can join even after the course has started. You can register and participate, although some early assignments may not be available to you. If you complete the required projects and peer reviews by the deadlines, you’ll still be eligible for a certificate.


## End to End RAG

In [35]:
import os

from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer = assistant.rag('I just discovered the course. Can I join now?')
print(answer)


Yes! You can join the course now. Just make sure to submit your project while we’re still accepting submissions if you want to receive a certificate.


In [36]:
custom_instructions = '''
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
'''.strip()

assistant = RAGBase(
    index=index,
    llm_client=client,
    instructions=custom_instructions,
)

In [37]:
assistant.rag('How do I get a certificate?')

'To earn a certificate, you must complete the course as part of a live cohort.  \nThe certificate is not awarded for the self‑paced version because it requires you to peer‑review three capstone projects while the cohort is active. Once the course is running, you’ll need to review those projects before you can receive the certificate.'

In [38]:
assistant.rag('Can I still join the course after it started?')

'Yes – you can still enroll after the course has started. However, to receive a certificate you must submit your project while we’re still accepting submissions.'

## Data Injection

- problem with minsearch
  - in-memory
  - it's bound to the process where it's running.
  - When you stop the process, the data disappears.
  - You need to re-index every time you restart.

- solution
  - seperate ingestion from querying
  - can use persistent search backend : Elasticsearch, OpenSearch, Qdrant, etc.

- sqlitesearch
  - a lightweight search library backed by SQLite FTS5.
  - It has the same API as minsearch, so switching is straightforward - it's a drop-in replacement, but persistent.

## Embeddings

all-MiniLM-L6-v2:

- 384-dimensional vectors (compact)
- Fast on CPU
- Good quality for general English text
- Uses cosine similarity 

In [39]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [40]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)

In [41]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [42]:
print(v1.shape)
print(dv.shape)
print(v1.dot(dv))

(384,)
(384,)
0.323324


In [43]:
v1.dot(dv)

np.float32(0.323324)

In [44]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [45]:
v2.dot(dv)

np.float32(0.019730432)

In [46]:
print(dv.shape)
print(dv)

(384,)
[-4.81206775e-02 -1.29942074e-01 -4.86808317e-03  1.37495529e-02
 -8.51100869e-03  6.97841570e-02 -7.55665125e-03  4.09571566e-02
 -1.02165632e-01 -3.45050246e-02  2.85521764e-02 -5.81430420e-02
 -1.85113382e-02 -3.17838937e-02  5.29631339e-02  3.96292657e-02
 -9.84595902e-03 -6.45324662e-02  7.05097467e-02 -6.39588907e-02
 -3.55156958e-02 -1.75490174e-02 -3.58401388e-02  5.50847617e-04
  3.38445231e-02  2.91763078e-02  3.24278362e-02 -5.64807141e-03
  8.42593312e-02  5.90306185e-02  2.08773017e-02 -5.44831250e-03
  2.98403483e-02  1.30820479e-02  6.36077598e-02 -3.06394100e-02
  6.28460050e-02 -1.69217885e-02 -4.65758219e-02 -5.13762161e-02
 -3.81560959e-02 -7.01197386e-02 -9.55517590e-02  5.01640476e-02
  8.57913122e-03  1.19207427e-02 -1.40597690e-02 -7.98978005e-03
 -3.44238505e-02 -2.34269928e-02  2.89465990e-02  2.68904790e-02
 -4.87408936e-02 -1.32924533e-02 -8.13224390e-02  4.34518754e-02
  6.60785958e-02 -4.70225848e-02 -2.56785173e-02 -9.07628089e-02
 -1.41221918e-02 -

The first score for q1 vs d (0.32) is higher, so the query is more similar to the document about registration.

The second score (q2 vs d) is lower - it's near 0. Installing Docker has nothing to do with registration.

- Cosine similarity measures the angle between two vectors, ignoring their length:
 - 1.0 = same direction (similar)
 - 0.0 = perpendicular (unrelated)
 - -1.0 = opposite direction (opposite meaning)
- Formally, if theta is the angle between two vectors, cosine similarity is cos(theta):
 - cos(0) = 1 - vectors point in the same direction
 - cos(90) = 0 - vectors are perpendicular
 - cos(180) = -1 - vectors point in opposite directions

### Loading the data

In [47]:
from ingest import load_faq_data

documents = load_faq_data()

### Generating embeddings

In [48]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [49]:
# for see the progress
from tqdm.auto import tqdm

In [50]:
# chunk the dataset into batches of 50 and encode the vectors:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [51]:
vectors[0].shape

(384,)

In [52]:
# We turn them into a 2-dimensional array (matrix) where
# rows are documents (vectors)
# columns are dimensions of the vectors
import numpy as np
X = np.array(vectors)

In [53]:
X.shape

(1208, 384)

# Vector Search

#### Encode

In [73]:
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

### A1 :  Vector Search by Compute dot product

In [74]:
scores = X.dot(v_query)

### Get Best match

In [75]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(553), np.float32(0.7629412))

In [76]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [77]:
#argsort --> sort the index by their scores ascending 
#[-5:]  --> top5 
top5 = np.argsort(scores)[-5:]

In [78]:
top5 = top5[::-1]
top5

array([553, 955,  29, 472, 558])

In [79]:
scores[top5]

array([0.7629412 , 0.7579371 , 0.7192131 , 0.65363115, 0.56009996],
      dtype=float32)

In [80]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629412
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

## A2 : Vector Search with Min Search

In [81]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [82]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [83]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [84]:
# filter by course

results = vindex.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

## Using RAGBase

In [85]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [86]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [88]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [89]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still sign up. However, if you’d like to receive a certificate, you’ll need to submit your project while we’re still accepting submissions.'

In [90]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [94]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [95]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still sign up. Just be sure to submit your project while we’re still accepting submissions if you want to receive a certificate.'

## A3:  Vector Search with sqlitesearch

In [96]:
!uv add sqlitesearch

Resolved 168 packages in 13ms
Checked 157 packages in 72ms


In [97]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

sqlitesearch supports three ANN modes:

| Mode | Best for | How it works |
|---|---|---|
| `lsh` (default) | Up to 100K vectors | Random hyperplane projections |
| `ivf` | 10K-500K vectors | K-means clustering |
| `hnsw` | 10K-1M+ vectors | Proximity graph (highest recall) |

All modes use two-phase search: approximate candidate retrieval, then exact cosine similarity reranking.

## Indexing the data

In [98]:
# Fit the index with our vectors and documents:

vs_index.fit(vectors, documents)

## Searching

In [99]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [100]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [101]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [102]:
vs_index.close()


## Reopening the index

In [103]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer('all-MiniLM-L6-v2')

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [104]:
query_vector = model.encode('How do I run Kafka?')
results = vs_index.search(query_vector, num_results=5)

In [ ]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=client,
)

In [107]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes – you can still join the program. However, to receive a certificate you’ll need to submit your project while submissions are still being accepted.'

In [108]:
vs_index.close()

## Comparing minsearch and sqlitesearch for vector search


| | minsearch `VectorSearch` | sqlitesearch `VectorSearchIndex` |
|---|---|---|
| Storage | In-memory (numpy) | Persistent (SQLite `.db` file) |
| Search | Exact cosine similarity | ANN (LSH/IVF/HNSW) + exact rerank |
| Startup | Must re-compute embeddings | Open existing index |
| Best for | Experiments, notebooks | Projects, persistence |

## Vector Search with PGVector

- PostgreSQL extension that adds vector similarity search to PostgreSQL
- handles concurrent access, transactions, and large datasets.

In [110]:
# Starting Postgres with pgvector

!docker run -it \
    --name pgvector \
    -e POSTGRES_USER=user \
    -e POSTGRES_PASSWORD=pswd \
    -e POSTGRES_DB=faq \
    -v pgvector_data:/var/lib/postgresql/data \
    -p 5432:5432 \
    pgvector/pgvector:pg17

/bin/bash: line 1: docker: command not found
